In [1]:
import os
import pandas as pd
import ants


# Paths
input_csv = 'E:/Datasets/corrupted data/selected_data1.csv'  # Path to the input CSV file
output_dir = 'E:/Datasets/Coregisterd-rigid'  # Directory to save co-registered data

# Load the MNI template
mni_template = ants.image_read("E:/Datasets/mni.nii.gz")  # MNI152 T1 template


# Load the CSV file
df = pd.read_csv(input_csv)

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)


# Loop through each subject
for index, row in df.iterrows():
    subject_id = row['Subject ID']
    print(subject_id)
    t1_path = row['T1 File Path']
    t2_path = row.get('T2 File Path' , None) # Some subjects may not have a T2
    seg_path = row.get('SEG File Path', None)  # Some subjects may not have a segmentation mask

    # Create subject-specific directories
    subject_dir = os.path.join(output_dir, subject_id)
    os.makedirs(subject_dir, exist_ok=True)
    t1_dir = os.path.join(subject_dir, 't1')
    t2_dir = os.path.join(subject_dir, 't2')
    seg_dir = os.path.join(subject_dir, 'seg')
    os.makedirs(t1_dir, exist_ok=True)
    os.makedirs(t2_dir, exist_ok=True)
    os.makedirs(seg_dir, exist_ok=True)

    t1_img = ants.image_read(t1_path)
    t1 = ants.reorient_image2(t1_img, orientation = "RAS")
    t1 = ants.n4_bias_field_correction(t1)
    

    # Step 1: Co-register T2 to T1
    if not pd.isna(t2_path) and (t2_path) != "":
        print("coregister t2 to t1")
        t2_img = ants.image_read(t2_path)
        t2 = ants.reorient_image2(t2_img, orientation = "RAS")
        t2 = ants.n4_bias_field_correction(t2)
        
        t2_to_t1 = ants.registration(
            fixed = t1,
            moving = t2,
            type_of_trasform = "Affine",
        )
        t2_registered_to_t1 = t2_to_t1["warpedmovout"]
        #registered_t2_to_t1_img = nib.Nifti1Image(registered_t2_to_t1_data, t1_img.affine, t1_img.header)
        #registered_t2_to_t1_path = os.path.join(t2_dir, 't2_to_t1.nii.gz')
        #nib.save(registered_t2_to_t1_img, registered_t2_to_t1_path)
    
    # Step 2: Co-register T1 to MNI
    print("coregister t1 to mni")
    t1_to_mni = ants.registration(
            fixed = mni_template,
            moving = t1,
            type_of_trasform = "rigid",
        )
    t1_registered_to_mni = t1_to_mni["warpedmovout"]
    registered_t1_to_mni_path = os.path.join(t1_dir, 't1_to_mni.nii.gz')
    ants.image_write(t1_registered_to_mni ,registered_t1_to_mni_path )

    # Step 3: Co-register T2 to MNI
    if not pd.isna(t2_path) and (t2_path) != "":
        print("coregister t2 to mni")
        t2_registered_to_mni = ants.apply_transforms(
            fixed = mni_template,
            moving = t2_registered_to_t1,
            transformlist = t1_to_mni["fwdtransforms"],
        )
        registered_t2_to_mni_path = os.path.join(t2_dir, 't2_to_mni.nii.gz')
        ants.image_write(t2_registered_to_mni, registered_t2_to_mni_path)
    else: 
        registered_t2_to_mni_path = None

    # Step 4: Apply T1-to-MNI transformation to segmentation mask (if it exists)
    if not pd.isna(seg_path) and (seg_path) != "":
        print("coregister seg to mni")
        seg_img = ants.image_read(seg_path)
        seg = ants.reorient_image2(seg_img, orientation = "RAS")
        
        seg_registered_to_mni = ants.apply_transforms(
            fixed = mni_template,
            moving = seg,
            transformlist = t1_to_mni["fwdtransforms"],
            interpolator = "nearestNeighbor"
        )
        registered_seg_to_mni_path = os.path.join(seg_dir, 'seg_to_mni.nii.gz')
        ants.image_write(seg_registered_to_mni, registered_seg_to_mni_path)
    else:
        registered_seg_to_mni_path = None

   
    print(f"Processing complete for subject {subject_id}")


print("All subjects processed.")

TCGA-08-0357
coregister t2 to t1
coregister t1 to mni
coregister t2 to mni
Processing complete for subject TCGA-08-0357
TCGA-08-0358
coregister t2 to t1
coregister t1 to mni
coregister t2 to mni
Processing complete for subject TCGA-08-0358
TCGA-08-0359
coregister t2 to t1
coregister t1 to mni
coregister t2 to mni
coregister seg to mni
Processing complete for subject TCGA-08-0359
TCGA-08-0360
coregister t2 to t1
coregister t1 to mni
coregister t2 to mni
coregister seg to mni
Processing complete for subject TCGA-08-0360
TCGA-08-0385
coregister t2 to t1
coregister t1 to mni
coregister t2 to mni
coregister seg to mni
Processing complete for subject TCGA-08-0385
TCGA-08-0389
coregister t2 to t1
coregister t1 to mni
coregister t2 to mni
coregister seg to mni
Processing complete for subject TCGA-08-0389
All subjects processed.
